# 02 Data Cleaning

Transform raw NYC restaurant inspection data into clean, analysis-ready tables.
Each cleaning step is written as a function so it can be reused in the pipeline later.

In [1]:
import pandas as pd

RAW_PATH = "../data/raw/restaurant_inspections_raw.csv"
PROCESSED_DIR = "../data/processed/"

raw = pd.read_csv(RAW_PATH, dtype=str)
print(raw.shape)

(294406, 27)


### Step 1: Standardize column names
Convert column names to lowercase snake_case (e.g., `INSPECTION DATE` → `inspection_date`) so they work cleanly in both Python and SQL.

In [2]:
def standardize_columns(df):
    """Convert column names to lowercase snake_case."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


df_named = standardize_columns(raw)
print(df_named.columns.tolist())

['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone', 'cuisine_description', 'inspection_date', 'action', 'violation_code', 'violation_description', 'critical_flag', 'score', 'grade', 'grade_date', 'record_date', 'inspection_type', 'latitude', 'longitude', 'community_board', 'council_district', 'census_tract', 'bin', 'bbl', 'nta', 'location']


### Step 2: Convert data types
Convert dates to datetime and score/coordinates to numeric. The next cell verifies that conversion did not create new missing values.

In [3]:
def convert_types(df):
    """Convert text columns to proper dates and numbers."""
    df = df.copy()

    for col in ["inspection_date", "grade_date"]:
        df[col] = pd.to_datetime(df[col], format="%m/%d/%Y", errors="coerce")

    for col in ["score", "latitude", "longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["zipcode"] = df["zipcode"].str.strip()
    return df


df_typed = convert_types(df_named)
print(df_typed[["inspection_date", "grade_date", "score", "latitude", "longitude"]].dtypes)

inspection_date    datetime64[ns]
grade_date         datetime64[ns]
score                     float64
latitude                  float64
longitude                 float64
dtype: object


In [4]:
# Validation: conversion should not create new missing values
for col in ["inspection_date", "grade_date", "score", "latitude", "longitude"]:
    before = df_named[col].isna().sum()
    after = df_typed[col].isna().sum()
    status = "OK" if before == after else "CHECK"
    print(f"{col:<16} missing before: {before:>7}   after: {after:>7}   {status}")

inspection_date  missing before:       0   after:       0   OK
grade_date       missing before:  160401   after:  160401   OK
score            missing before:   17389   after:   17389   OK
latitude         missing before:    1666   after:    1666   OK
longitude        missing before:    1666   after:    1666   OK


### Step 3: Fix disguised missing values (Issues #3, #4, #14)
Replace placeholder values that actually mean "missing" with NaN: borough `"0"` and coordinates `(0, 0)`.

In [5]:
import numpy as np

def fix_disguised_missing(df):
    """Replace placeholder values that actually mean 'missing' with real NaN."""
    df = df.copy()

    # Borough "0" is not a valid NYC borough
    df["boro"] = df["boro"].replace("0", np.nan)

    # Coordinates (0, 0) are not in NYC
    zero_coords = (df["latitude"] == 0) | (df["longitude"] == 0)
    df.loc[zero_coords, ["latitude", "longitude"]] = np.nan

    return df


df_fixed = fix_disguised_missing(df_typed)

In [6]:
# Validation 1: borough
print("boro = '0' before:", (df_typed["boro"] == "0").sum())
print("boro = '0' after: ", (df_fixed["boro"] == "0").sum())
print("boro missing after:", df_fixed["boro"].isna().sum())

# Validation 2: coordinates
zero_before = ((df_typed["latitude"] == 0) | (df_typed["longitude"] == 0)).sum()
missing_before = df_typed["latitude"].isna().sum()
missing_after = df_fixed["latitude"].isna().sum()

print("\nzero coordinates before:", zero_before)
print("expected missing after:", missing_before + zero_before)
print("actual missing after:  ", missing_after)
print("Match:", missing_before + zero_before == missing_after)

boro = '0' before: 311
boro = '0' after:  0
boro missing after: 311

zero coordinates before: 3085
expected missing after: 4751
actual missing after:   4751
Match: True


### Step 4: Fill missing borough (Issues #5, #15, #16)
Fill missing borough from (1) the same restaurant's other rows, then (2) the most common borough for the same ZIP code. Filled rows are flagged in `boro_imputed`.

In [10]:
def fill_borough(df):
    """Fill missing borough using (1) the same restaurant's other rows, then (2) the most common borough for the same ZIP code."""
    df = df.copy()
    originally_missing = df["boro"].isna()

    # Step 1: borrow from the same restaurant
    boro_by_camis = (
        df.dropna(subset=["boro"])
        .groupby("camis")["boro"]
        .agg(lambda s: s.mode().iloc[0])
    )
    df["boro"] = df["boro"].fillna(df["camis"].map(boro_by_camis))

    # Step 2: borrow from restaurants in the same ZIP code
    boro_by_zip = (
        df.dropna(subset=["boro", "zipcode"])
        .groupby("zipcode")["boro"]
        .agg(lambda s: s.mode().iloc[0])
    )
    df["boro"] = df["boro"].fillna(df["zipcode"].map(boro_by_zip))

    # Flag rows that were missing and successfully filled
    df["boro_imputed"] = originally_missing & df["boro"].notna()

    return df


df_boro = fill_borough(df_fixed)

In [11]:
print("boro missing before:", df_fixed["boro"].isna().sum())
print("boro missing after: ", df_boro["boro"].isna().sum())
print("rows imputed:", df_boro["boro_imputed"].sum())

print("\n", df_boro["boro"].value_counts(dropna=False))

# Consistency check: each restaurant should belong to exactly one borough
boro_per_camis = df_boro.groupby("camis")["boro"].nunique()
print("\nrestaurants with more than one borough:", (boro_per_camis > 1).sum())

boro missing before: 311
boro missing after:  268
rows imputed: 43

 boro
Manhattan        109009
Brooklyn          74430
Queens            73836
Bronx             27418
Staten Island      9445
NaN                 268
Name: count, dtype: int64

restaurants with more than one borough: 0


In [12]:
# Why couldn't these be filled?
still_missing = df_boro["boro"].isna()
print("still missing:", still_missing.sum())
print("  of which also missing zipcode:", df_boro.loc[still_missing, "zipcode"].isna().sum())
print("  unique restaurants affected:", df_boro.loc[still_missing, "camis"].nunique())

still missing: 268
  of which also missing zipcode: 69
  unique restaurants affected: 96


In [9]:
# Chains: one brand name (dba) can have many locations (camis)
locations_per_name = df_boro.groupby("dba")["camis"].nunique().sort_values(ascending=False)
print(locations_per_name.head(10))

# Each camis should map to exactly one address
address_per_camis = df_boro.groupby("camis")[["building", "street"]].nunique()
print("\ncamis with more than one address:", ((address_per_camis > 1).any(axis=1)).sum())

dba
DUNKIN                    333
STARBUCKS                 168
MCDONALD'S                166
SUBWAY                    162
POPEYES                   121
DUNKIN'                   111
DOMINO'S                   73
CHIPOTLE MEXICAN GRILL     72
TACO BELL                  70
KENNEDY FRIED CHICKEN      68
Name: camis, dtype: int64

camis with more than one address: 0


### Step 5: Add inspection fields and analysis scope (Issues #2, #8, #18, #19)
Split `inspection_type` into program and stage, flag never-inspected restaurants (placeholder date 1900-01-01), and mark in-scope inspections in `in_scope`.

In [13]:
SCOPE_PROGRAMS = ["Cycle Inspection", "Pre-permit (Operational)"]
SCOPE_STAGES = ["Initial Inspection", "Re-inspection"]

def add_inspection_fields(df):
    """Split inspection type, flag never-inspected restaurants, and mark in-scope inspections."""
    df = df.copy()

    # Split "Program / Stage" into two columns
    split = df["inspection_type"].str.split(" / ", n=1, expand=True)
    df["inspection_program"] = split[0]
    df["inspection_stage"] = split[1]

    # Flag restaurants that have never been inspected (placeholder date)
    df["never_inspected"] = df["inspection_date"] == pd.Timestamp("1900-01-01")

    # Mark inspections within our analysis scope
    df["in_scope"] = (
        df["inspection_program"].isin(SCOPE_PROGRAMS)
        & df["inspection_stage"].isin(SCOPE_STAGES)
        & ~df["never_inspected"]
    )
    return df


df_insp = add_inspection_fields(df_boro)

In [14]:
# Check 1: never-inspected count should be 3,917
print("never inspected rows:", df_insp["never_inspected"].sum())

# Check 2: every inspection_type should split into exactly two parts
bad_split = df_insp["inspection_type"].notna() & df_insp["inspection_stage"].isna()
print("rows that failed to split:", bad_split.sum())

# Check 3: what does the split look like?
print("\n", df_insp["inspection_program"].value_counts(dropna=False))
print("\n", df_insp["inspection_stage"].value_counts(dropna=False))

# Check 4: size of the analysis scope
print("\nin-scope rows:", df_insp["in_scope"].sum())
print("in-scope inspections:",
      df_insp.loc[df_insp["in_scope"], ["camis", "inspection_date", "inspection_type"]].drop_duplicates().shape[0])

never inspected rows: 3917
rows that failed to split: 0

 inspection_program
Cycle Inspection                  217956
Pre-permit (Operational)           53616
Administrative Miscellaneous       11574
Pre-permit (Non-operational)        4005
NaN                                 3917
Inter-Agency Task Force             1397
Smoke-Free Air Act                   728
Calorie Posting                      576
Trans Fat                            503
Sodium Warning                        91
Accelerated Inspection Program        43
Name: count, dtype: int64

 inspection_stage
Initial Inspection              213426
Re-inspection                    69626
NaN                               3917
Compliance Inspection             3730
Reopening Inspection              3412
Second Compliance Inspection       295
Name: count, dtype: int64

in-scope rows: 264489
in-scope inspections: 77125


In [15]:
# Exact duplicate rows
print("exact duplicate rows:", df_insp.duplicated().sum())

# Inspection-level consistency: one inspection should have one score, one grade, one action
key = ["camis", "inspection_date", "inspection_type"]
in_scope = df_insp[df_insp["in_scope"]]

per_insp = in_scope.groupby(key).agg(
    n_scores=("score", "nunique"),
    n_grades=("grade", "nunique"),
    n_actions=("action", "nunique"),
)
print("\ninspections with more than one score: ", (per_insp["n_scores"] > 1).sum())
print("inspections with more than one grade: ", (per_insp["n_grades"] > 1).sum())
print("inspections with more than one action:", (per_insp["n_actions"] > 1).sum())

# Violation-level: is the same violation code listed twice in one inspection?
repeated_codes = in_scope.duplicated(subset=key + ["violation_code"]).sum()
print("\nrepeated violation codes within an inspection:", repeated_codes)

exact duplicate rows: 217

inspections with more than one score:  0
inspections with more than one grade:  0
inspections with more than one action: 0

repeated violation codes within an inspection: 206


In [16]:
# Remove exact duplicates first, then check again
in_scope_dedup = in_scope.drop_duplicates()
print("in-scope rows before:", len(in_scope))
print("in-scope rows after removing exact duplicates:", len(in_scope_dedup))
print("rows removed:", len(in_scope) - len(in_scope_dedup))

remaining = in_scope_dedup.duplicated(subset=key + ["violation_code"]).sum()
print("\nrepeated violation codes remaining:", remaining)

in-scope rows before: 264489
in-scope rows after removing exact duplicates: 264283
rows removed: 206

repeated violation codes remaining: 0


### Steps 6–7: Remove exact duplicates and flag grade/score mismatches (Issues #13, #20, #22)
Drop rows that are exact copies (no distinguishing information), then flag A/B/C grades that do not match the score band in `grade_score_mismatch`.

In [17]:
def remove_exact_duplicates(df):
    """Drop rows that are exact copies of another row."""
    before = len(df)
    df = df.drop_duplicates().copy()
    print(f"remove_exact_duplicates: dropped {before - len(df)} rows")
    return df


def flag_grade_score_mismatch(df):
    """Flag rows where an A/B/C grade does not match the score band."""
    df = df.copy()
    expected = pd.cut(df["score"], bins=[-1, 13, 27, 1000], labels=["A", "B", "C"]).astype(str)
    df["grade_score_mismatch"] = df["grade"].isin(["A", "B", "C"]) & (df["grade"] != expected)
    return df


df_dedup = remove_exact_duplicates(df_insp)
df_clean = flag_grade_score_mismatch(df_dedup)

remove_exact_duplicates: dropped 217 rows


In [18]:
# Check 1: dropped rows should be 217 (all data, not just in scope)
print("rows after dedup:", len(df_clean))

# Check 2: mismatch at inspection level, in scope
mismatch_insp = (
    df_clean.loc[df_clean["in_scope"] & df_clean["grade_score_mismatch"], key]
    .drop_duplicates()
    .shape[0]
)
print("in-scope inspections with grade/score mismatch:", mismatch_insp)

rows after dedup: 294189
in-scope inspections with grade/score mismatch: 28
